# Figure 03 — functional ultrasound responses to visual stimuli

Reproduction workflow for NTSRp **left-hemisphere** responses to looming (LT), dimming (DT), and sweeping (SB), including paired Vehicle–CNO comparisons.

1. Install the dependencies and arrange input files as described in **README.md**.
2. Edit the paths and parameters below.
3. Select **Restart Kernel and Run All**. Figures (PNG/SVG), statistical tables, and a run manifest are written to `results/`.

Run from the folder containing this notebook and its two helper modules. The full bootstrap analysis can take several minutes. Saved outputs are cleared deliberately so they cannot be confused with a fresh run. The original notebook is unchanged.

**Methods retained:** median baseline samples `[4:30]`; activity mean `[31:45]`; CNO mean `[31:35]`; one-sample Cohen’s d; sign-flip permutation tests; percentile bootstrap intervals; Benjamini–Hochberg correction. Slices use zero-based indices with the final index excluded. See README for inferential limitations and documented corrections.


## 1. Configuration


In [ ]:
from pathlib import Path
import os

# Launch Jupyter from this folder. Alternatively set FUSI_PROJECT_DIR.
PROJECT_DIR = Path(os.environ.get("FUSI_PROJECT_DIR", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("FUSI_DATA_DIR", PROJECT_DIR / "data/tracesPerAnimalB"))
LABELS_FILE = Path(os.environ.get("FUSI_LABELS_FILE", PROJECT_DIR / "data/RegionsListLabels.txt"))
ATLAS_DIR = Path(os.environ.get("FUSI_ATLAS_DIR", PROJECT_DIR / "data/atlas"))
OUTPUT_DIR = Path(os.environ.get("FUSI_OUTPUT_DIR", PROJECT_DIR / "results"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full analysis defaults retained from the source notebook.
RANDOM_SEED = 42
N_PERMUTATIONS = 99_999
N_BOOTSTRAPS_ACTIVITY = 50_000
N_BOOTSTRAPS_CNO = 5_000
FDR_ALPHA = 0.05
ACTIVITY_HIGHLIGHT_ALPHA = 0.01  # Panel A used a stricter display threshold.
BASELINE = slice(4, 30)
ACTIVITY_WINDOW = slice(31, 45)
CNO_WINDOW = slice(31, 35)
TRACE_SIGMA = 0.1
PCA_SIGMA = 1.5
PCA_START = 14
STIM_START = 30
STIM_END = {"Looming": 41, "Dimming": 41, "Sweeping": 46}
SAMPLE_INTERVAL_S = 0.5  # Source axis labels imply 2 Hz; see README.
FONT_FAMILY = "DejaVu Sans"  # Bundled with Matplotlib for portability.
SIGNIFICANT_ONLY = False
# User confirmed Vehicle and CNO have matching animals in matching order.
PAIRED_ANIMALS_CONFIRMED = True


## 2. Dependencies and validated inputs


In [ ]:
# System
from pathlib import Path
import sys

# Numbers & Data
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
from IPython.display import display

# Analysis & Statistics
from scipy.io import loadmat
from scipy import stats
from scipy.ndimage import median_filter, gaussian_filter1d
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests
import dabest
import json
import hashlib
import platform
import importlib.metadata
from datetime import datetime, timezone

# Local helpers must be found before importing them.
if not (PROJECT_DIR / "atlas_tools.py").is_file():
    raise FileNotFoundError("Launch Jupyter from the folder containing atlas_tools.py.")
sys.path.insert(0, str(PROJECT_DIR))
from analysis_helpers import run_estimation_and_permutation_test

# Brain Mapping Tools
from atlas_tools import (
    atlas_plane,
    draw_borders,
    load_matlab_atlas,
    mat_field,
    read_file_list,
    recolor_map_double,
)


plt.rcParams["svg.hashsalt"] = "fusi-figure-03"

def save_figure(figure, filename, **kwargs):
    """Save every figure in the configured results directory."""
    axes_3d = [ax for ax in figure.axes if getattr(ax, "name", "") == "3d"]
    if axes_3d:
        figure.canvas.draw()
        kwargs["bbox_extra_artists"] = [label for ax in axes_3d for label in
                                        (ax.xaxis.label, ax.yaxis.label, ax.zaxis.label)]
        kwargs["pad_inches"] = 0.15
    figure.savefig(OUTPUT_DIR / filename, **kwargs)


In [ ]:
# Validate all required files before starting the expensive analysis.
required = [LABELS_FILE, ATLAS_DIR / "atlasPermutedmmV02.mat",
            ATLAS_DIR / "listSelectedRegionsV02.txt"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing input files (see README):\n" + "\n".join(missing))
region_labels = LABELS_FILE.read_text().splitlines()
if len(region_labels) != 319 or any(not label.strip() for label in region_labels):
    raise ValueError("Expected exactly 319 nonempty region labels in acquisition order.")
input_files = list(required)
input_manifest = []

def load_condition(stimulus, treatment):
    suffix = f"NTSRp_{stimulus}_{treatment}"
    files = sorted(DATA_DIR.glob(f"*/{suffix}.mat"))
    files = sorted(set(files + list(DATA_DIR.glob(f"*/*_{suffix}.mat"))))
    if not files:
        raise FileNotFoundError(f"No *{suffix}.mat files under {DATA_DIR}/<group>/")
    arrays = []
    for path in files:
        content = loadmat(path)
        if "PL" not in content:
            raise KeyError(f"{path.name}: missing PL (left-hemisphere traces)")
        array = np.asarray(content["PL"], dtype=float)
        if array.ndim != 3 or array.shape[0] != 319 or array.shape[1] < 70 or array.shape[2] < 2:
            raise ValueError(f"{path.name}: expected (319, >=70, >=2), got {array.shape}")
        if not np.isfinite(array).all():
            raise ValueError(f"{path.name}: traces contain NaN or infinite values")
        arrays.append(array)
        input_files.append(path)
        input_manifest.append({"file": str(path.relative_to(DATA_DIR)),
                               "stimulus": stimulus, "treatment": treatment,
                               "shape": list(array.shape)})
    if len({a.shape[:2] for a in arrays}) != 1:
        raise ValueError(f"Incompatible trace shapes for {suffix}")
    return np.concatenate(arrays, axis=2)

def smooth_trace(data, sigma=TRACE_SIGMA):
    """Median-filter five samples, then Gaussian-filter the time axis."""
    return gaussian_filter1d(median_filter(data, size=(1, 5)), sigma=sigma, axis=1)

LoomT_V, LoomT_C = load_condition("LT", "V"), load_condition("LT", "C")
DimmT_V, DimmT_C = load_condition("DT", "V"), load_condition("DT", "C")
SweepT_V, SweepT_C = load_condition("SB", "V"), load_condition("SB", "C")
if not PAIRED_ANIMALS_CONFIRMED:
    raise ValueError("Confirm animal identity/order before paired CNO analysis.")
for stimulus, vehicle, cno in [("LT", LoomT_V, LoomT_C), ("DT", DimmT_V, DimmT_C), ("SB", SweepT_V, SweepT_C)]:
    if vehicle.shape != cno.shape:
        raise ValueError(f"{stimulus}: Vehicle/CNO shapes differ; pairing is invalid")
    pairs = {t: [item for item in input_manifest if item['stimulus'] == stimulus and item['treatment'] == t] for t in ['V', 'C']}
    v_keys = [(x['file'][:-5], x['shape']) for x in pairs['V']]
    c_keys = [(x['file'][:-5], x['shape']) for x in pairs['C']]
    if v_keys != c_keys:
        raise ValueError(f"{stimulus}: source file blocks do not match across conditions")
if len({a.shape[1] for a in [LoomT_V, DimmT_V, SweepT_V]}) != 1:
    raise ValueError("Stimuli must have matching numbers of time samples")
display(pd.DataFrame(input_manifest))

LoomL_raw, DimL_raw, SweepL_raw = LoomT_V, DimmT_V, SweepT_V
LoomL, DimL, SweepL = [a.mean(axis=2) for a in [LoomL_raw, DimL_raw, SweepL_raw]]
LoomL_smooth, DimL_smooth, SweepL_smooth = [smooth_trace(a) for a in [LoomL, DimL, SweepL]]
aLT, aDT, aSB = [a[:, ACTIVITY_WINDOW, :].mean(axis=1) - np.median(a[:, BASELINE, :], axis=1)
                 for a in [LoomL_raw, DimL_raw, SweepL_raw]]
aLTveh, aDTveh, aSBveh = [a[:, CNO_WINDOW, :].mean(axis=1) - np.median(a[:, BASELINE, :], axis=1)
                        for a in [LoomT_V, DimmT_V, SweepT_V]]
aLTcno, aDTcno, aSBcno = [a[:, CNO_WINDOW, :].mean(axis=1) - np.median(a[:, BASELINE, :], axis=1)
                        for a in [LoomT_C, DimmT_C, SweepT_C]]

atlas = load_matlab_atlas(ATLAS_DIR / "atlasPermutedmmV02.mat")
atlas_regions = np.asarray(mat_field(atlas, "Regions")) - 1
selection = read_file_list(ATLAS_DIR / "listSelectedRegionsV02.txt", mat_field(atlas, "infoRegions"))
if len(selection) != 319:
    raise ValueError(f"Expected 319 atlas selections, found {len(selection)}")
label_mismatches = pd.DataFrame([
    {"region_index": i, "trace_label": trace, "atlas_label": mapped}
    for i, (trace, mapped) in enumerate(zip(region_labels, selection.acr)) if trace != mapped
], columns=["region_index", "trace_label", "atlas_label"])
label_mismatches.to_csv(OUTPUT_DIR / "atlas_label_mismatches.csv", index=False)
if not label_mismatches.empty:
    print("REVIEW: trace/atlas labels differ; retaining original positional mapping.")
    display(label_mismatches)
print(f"Atlas shape: {atlas_regions.shape}; selected regions: {len(selection)}")


# Panel A - Heatmap of Responses to Visual Stimuli


### Define Area Names for Analysis and Plotting


In [ ]:
#---------- Define Area Names For Plotting ----------#
#Structure borders index, indexes are before removing any area from list.
CTXpl = np.arange(0, 45)    # cortical plate
CTXspl = np.arange(45, 115) # cortical subplate
TH = np.arange(115, 160)    # Thalamus
HY = np.arange(160, 190)    # Hypothalamus
MB = np.arange(190, 250)    # Midbrain
HB_C = np.arange(250, 319)  # Hindbrain and cerebellum

#List of Grouped Areas for Labeling
ACA = np.arange(1, 3)
AUD = np.arange(4, 8)
RSP = np.arange(18, 21)
SSp = np.arange(21, 28)
VIS = np.arange(32, 45)
COA = np.arange(50, 53)
BLA = np.arange(54, 57)
BMA = np.arange(57, 59)
CEA = np.arange(59, 62)
HIPP = np.arange(66, 78)
SUBp = np.arange(80, 84)
CPc = np.arange(87, 92)
CPm = np.arange(92, 101)
GP = np.arange(102, 104)
LS = np.arange(104, 107)
LG = np.arange(128, 131)
MG = np.arange(133, 136)
PTR = np.arange(195, 200)
MLR = np.arange(201, 203)
IC = np.arange(206, 209)
MRN = np.arange(212, 216)
PAG = np.arange(221, 228)
SC = np.arange(236, 246)
SN = np.arange(246, 248)

# List of areas not to plot
NoplotIndexes = np.array([1, 9, 65, 111, 114, 116, 161, 191, 142, 251, 265, 297, 311, 312, 313, 314, 315, 316, 317, 318, 319])-1 

# Pathways
# --> SCs, LP, CPc, LA, VISpor (putative WF pathways)
SCs = np.arange(242, 246)
LP = 131
CPc = np.arange(87, 90)
VISpor = 43

# --> SCs, SCi, PAGd, CUN (putative NF pathway)
SCs = np.arange(242, 246)
SCi = np.arange(236, 243)
PAGd = np.array([222, 225])
CUN = 202

# --> LGd, VISp(pl), VISl, VISli, VISpl, VISpor, TEa (Ventral stream "what")
LGd = 128
VISv = np.array([39, 35, 36, 41, 43, 29])

# --> LGd, VISp(al), VISp(am), VISp(pm), VISal, VISam, VISpm, VISrl, RSPagl, RSPd, APr, LD, POST, CA1d, CA3d (dorsal stream "where")
LGd = 128
VISd = np.array([37, 38, 40, 33, 34, 42, 44, 18, 19, 9, 127, 81, 66, 72])


### Activity statistics

Positive responses use a one-sided `greater` test. FDR excludes the listed non-plotted regions. Panel A highlights q < 0.01; counts use q < 0.05. Exploratory metric sweeps are archived in `exploratory_notes.md` and cannot overwrite these results.


In [ ]:
# ---------- Run Pipeline for Each Condition ---------- #
chosen_metric = 'cohens_d'  
nsamples = N_PERMUTATIONS
nboots = N_BOOTSTRAPS_ACTIVITY
ALT = 'greater'
RS = RANDOM_SEED
all_results = {}

res_LT = run_estimation_and_permutation_test(aLT, effect_type=chosen_metric, alpha=FDR_ALPHA, n_resamples=nsamples, n_bootstraps=nboots, no_plot_indexes=NoplotIndexes, alternative=ALT, random_state=RS)
res_DT = run_estimation_and_permutation_test(aDT, effect_type=chosen_metric, alpha=FDR_ALPHA, n_resamples=nsamples, n_bootstraps=nboots, no_plot_indexes=NoplotIndexes, alternative=ALT, random_state=RS)
res_SB = run_estimation_and_permutation_test(aSB, effect_type=chosen_metric, alpha=FDR_ALPHA, n_resamples=nsamples, n_bootstraps=nboots, no_plot_indexes=NoplotIndexes, alternative=ALT, random_state=RS)

# Unpack Workflow Variables
pLT, apLT, sig_LT = res_LT['p_raw'], res_LT['p_corrected'], res_LT['sig_rows']
pDT, apDT, sig_DT = res_DT['p_raw'], res_DT['p_corrected'], res_DT['sig_rows']
pSB, apSB, sig_SB = res_SB['p_raw'], res_SB['p_corrected'], res_SB['sig_rows']

# Extract active areas
num_active_LT = len(res_LT['sig_rows'])
num_active_DT = len(res_DT['sig_rows'])
num_active_SB = len(res_SB['sig_rows'])
all_results[chosen_metric] = {'LT': num_active_LT, 'DT': num_active_DT, 'SB': num_active_SB}

print(f"Active areas (LT): {num_active_LT}")
print(f"Active areas (DT): {num_active_DT}")
print(f"Active areas (SB): {num_active_SB}")


In [ ]:
unique_active_areas = np.unique(np.concatenate([
    res_LT["sig_rows"],
    res_DT["sig_rows"],
    res_SB["sig_rows"],
]))

total_unique_active_areas = len(unique_active_areas)

print(f"Total unique active areas: {total_unique_active_areas}")


In [ ]:
active_in_all_three = np.intersect1d(
    np.intersect1d(res_LT["sig_rows"], res_DT["sig_rows"]),
    res_SB["sig_rows"]
)

num_active_in_all_three = len(active_in_all_three)

print(f"Areas active in all three stimuli: {num_active_in_all_three}")


### Plot HeatMap


In [ ]:
savefig = True

# ---------- Define Figures and Axes Layout ---------- #
width_mm = 130
height_mm = 150
fig = plt.figure(figsize=(width_mm / 25.4, height_mm / 25.4))

sns.set_theme(font=FONT_FAMILY, style="white")

gs = fig.add_gridspec(2, 6, height_ratios=[1, 0.08], hspace=0.3, wspace=0.2)
c_map = sns.diverging_palette(220, 20, sep=30, as_cmap=True)

# Create axes pairs for Looming, Dimming, and Sweeping
axes = [fig.add_subplot(gs[0, i]) for i in range(6)]
ax1, ax2, ax3, ax4, ax5, ax6 = axes

# ---------- Organize Data ---------- #
keep_idx = np.setdiff1d(np.arange(LoomL_smooth.shape[0]), NoplotIndexes)

def extract_stim(data_smooth, ap, res):
    return (
        data_smooth[keep_idx],
        ap[keep_idx],
        res['effect_size'][keep_idx],
        res['ci_lower'][keep_idx],
        res['ci_upper'][keep_idx],
    )

Loom, apLT_plot, eff_LT, ci_low_LT, ci_high_LT = extract_stim(LoomL_smooth, apLT, res_LT)
Dimm, apDT_plot, eff_DT, ci_low_DT, ci_high_DT = extract_stim(DimL_smooth, apDT, res_DT)
Sweep, apSB_plot, eff_SB, ci_low_SB, ci_high_SB = extract_stim(SweepL_smooth, apSB, res_SB)

# Major brain-region segments matching Allen Brain Atlas official colors
region_color_map = {
    "CTX": "#70F470",   # Isocortex
    "CTXs": "#40E0D0",  # Cortical Subplate
    "TH": "#FF8080",    # Thalamus
    "HY": "#E60000",    # Hypothalamus
    "MB": "#FF70F7",    # Midbrain
    "HBC": "#FF9900",   # Hindbrain & Cerebellum
}

region_defs = [
    (CTXpl, region_color_map["CTX"]),
    (CTXspl, region_color_map["CTXs"]),
    (TH, region_color_map["TH"]),
    (HY, region_color_map["HY"]),
    (MB, region_color_map["MB"]),
    (HB_C, region_color_map["HBC"]),
]

region_segments = []
for orig_indices, color in region_defs:
    mask = np.isin(orig_indices, keep_idx)
    filtered_indices = np.where(mask)[0]
    if len(filtered_indices) > 0:
        y_start = np.searchsorted(keep_idx, orig_indices[filtered_indices[0]])
        y_end = np.searchsorted(keep_idx, orig_indices[filtered_indices[-1]]) + 1
        region_segments.append((y_start, y_end, color))

# ---------- Define Grouped Area Labels ---------- #
grouped_areas = {
    "ACA": np.arange(1, 3), "AUD": np.arange(4, 8), "RSP": np.arange(18, 21),
    "SSp": np.arange(21, 28), "VIS": np.arange(32, 45), "AMG": np.arange(54, 62), 
    "HIPP": np.arange(66, 78), "SUBp": np.arange(80, 84), 
    "CPc": np.arange(87, 92), "CPm": np.arange(92, 101), "GP": np.arange(102, 104),
    "LG": np.arange(128, 131), "MG": np.arange(133, 136), "PTR": np.arange(195, 200),
    "MLR": np.arange(201, 203), "MRN": np.arange(212, 216), "PAG": np.arange(221, 228), 
    "SC": np.arange(236, 246), "SN": np.arange(246, 248)
}

threshold = ACTIVITY_HIGHLIGHT_ALPHA

def compute_extents(p_plot):
    extents = []
    for label, orig_indices in grouped_areas.items():
        mask = np.isin(orig_indices, keep_idx)
        filtered_indices = np.where(mask)[0]
        if len(filtered_indices) > 0:
            y_positions = np.searchsorted(keep_idx, orig_indices[filtered_indices])
            y_min, y_max = y_positions[0] - 0.5, y_positions[-1] + 0.5
            if np.mean(p_plot[y_positions] < threshold) > 0.25:
                extents.append((y_min, y_max, np.mean([y_min, y_max]), label))
    return extents

# Compute extents based purely on Looming's p-values
group_extents_loom = compute_extents(apLT_plot)

# ---------- Plot Heatmaps ---------- #
heatmap_configs = [
    (ax1, Loom, "Looming", 41),
    (ax3, Dimm, "Dimming", 41),
    (ax5, Sweep, "Sweeping", 46),
]

for ax, data, title, end_frame in heatmap_configs:
    hm = ax.imshow(data, cmap=c_map, vmin=-0.01, vmax=0.01, aspect="auto", origin="upper")
    
    # Only draw the major brain-region color bars on the Looming heatmap (ax1)
    if ax == ax1:
        for y0, y1, color in region_segments:
            ax.axhspan(y0 - 0.5, y1 - 0.5, xmin=-0.06, xmax=0.0, facecolor=color, edgecolor='none', zorder=20, clip_on=False)

    ax.axvline(STIM_START, color=".4", linewidth=1, label="StimulusStart")
    ax.axvline(end_frame, color=".4", linewidth=1, linestyle="--", label="StimulusEnd")
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.set_xticks([20, 30, 60])
    ax.set_xticklabels([f"{(frame - STIM_START) * SAMPLE_INTERVAL_S:g}" for frame in [20, 30, 60]], fontsize=6)
    ax.set_xlabel("Time (s)", fontsize=7)

# ---------- Plot Effect Size with CIs ---------- #
y_vals = np.arange(Loom.shape[0])

# Pass group_extents_loom to ALL effect configs so Dimming and Sweeping share Looming's brackets and labels
effect_configs = [
    (ax2, eff_LT, ci_low_LT, ci_high_LT, apLT_plot, group_extents_loom),
    (ax4, eff_DT, ci_low_DT, ci_high_DT, apDT_plot, group_extents_loom),
    (ax6, eff_SB, ci_low_SB, ci_high_SB, apSB_plot, group_extents_loom)
]

y_limits_exact = (Loom.shape[0]  - 0.5, -0.5)

# Style Heatmaps
for ax in [ax1, ax3, ax5]:
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_yticks([])
    ax.set_ylim(y_limits_exact)
    ax.set_xlim(20, 70)
    ax.tick_params(axis="both", labelsize=6)

# Style Effect Size Axes and Draw Extent Brackets
for ax, eff, ci_low, ci_high, p_vals, extents in effect_configs:
    for y, e, l, h, p in zip(y_vals, eff, ci_low, ci_high, p_vals):
        color = "indianred" if p < threshold else "black"
        alpha = 0.8 if p < threshold else 0.15
        ax.plot([l, h], [y, y], color=color, alpha=alpha, linewidth=.5)
        ax.scatter([e], [y], color=color, alpha=alpha, s=2)
        
    ax.axvline(0, color=".4", linewidth=0.8, linestyle=":")
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylim(y_limits_exact)
    ax.set_xlim(-1, 6)
    ax.set_yticks([])
  
    x_min_lim, x_max_lim = ax.get_xlim()
    for y_min, y_max, y_center, label in extents:
        ax.vlines(x=x_max_lim * 1.6, ymin=y_min, ymax=y_max, color='gray', linewidth=1.5, transform=ax.transData, clip_on=False)
        ax.text(x=x_max_lim * 2, y=y_center, s=label, color='black', fontsize=6, va='center', ha='left', transform=ax.transData, clip_on=False)
      
    ax.tick_params(axis="x", labelsize=6)
    ax.set_xlabel("Cohen’s $d$", fontsize=7)
    
    pos = ax.get_position()
    ax.set_position([pos.x0 + 0.0, pos.y0, pos.width * 0.4, pos.height])

# ---------- Single Color Bar Under Dimming Stimulus ---------- #
cax = fig.add_subplot(gs[1, 2:4])
cbar = fig.colorbar(ax3.images[0], cax=cax, orientation="horizontal")
cbar.set_label("Activity Amplitude", fontsize=8)
cbar.ax.tick_params(labelsize=6)
pos = cbar.ax.get_position()
cbar.ax.set_position([pos.x0, pos.y0 + 0.04, pos.width, pos.height * 0.25])
cbar.outline.set_visible(False)
cbar.set_ticks([])

# ---------- DISPLAY and SAVE ---------- #
savefig = True
if savefig:
    save_figure(plt.gcf(), "Panel_A.png", dpi=300, bbox_inches="tight")
    save_figure(plt.gcf(), "Panel_A.svg", bbox_inches="tight")

plt.show()
plt.close("all")


# Panel B - PCA Plot of Visual Responses


### Calculate the PCA Space


In [ ]:
# Define areas to exclude (converting your list to 0-based indexing)
NoplotIndexes = np.array([1, 9, 65, 111, 114, 116, 161, 191, 142, 251, 265, 297, 311, 312, 313, 314, 315, 316, 317, 318, 319]) - 1

# Create a keep mask or use setdiff1d to filter the rows of your initial data
keep_idx = np.setdiff1d(np.arange(DimL.shape[0]), NoplotIndexes)

# Filter the raw/base data matrices to drop excluded areas
DimL_filtered = DimL[keep_idx, :]
LoomL_filtered = LoomL[keep_idx, :]
SweepL_filtered = SweepL[keep_idx, :]

#---------- Medians ---------#
# ----- Dimming ----- #
DimL_median = median_filter(DimL_filtered, size=(1, 5))

# ----- Looming ----- #
LoomL_median = median_filter(LoomL_filtered, size=(1, 5))

# ----- Sweeping ----- #
SweepL_median = median_filter(SweepL_filtered, size=(1, 5))

#---------- Smooth PCA ---------- #
sigma = PCA_SIGMA  # smoothing strength

# ----- Dimming ----- #
DimL_smooth2 = gaussian_filter1d(DimL_median, sigma=sigma, axis=1)

# ----- Looming ----- #
LoomL_smooth2 = gaussian_filter1d(LoomL_median, sigma=sigma, axis=1)

# ----- Sweeping ----- #
SweepL_smooth2 = gaussian_filter1d(SweepL_median, sigma=sigma, axis=1)

DIM = DimL_smooth2[:, PCA_START:]  # Python uses 0-based indexing (15:end becomes PCA_START:)
LOOM = LoomL_smooth2[:, PCA_START:]
SWEEP = SweepL_smooth2[:, PCA_START:]

# Combine arrays horizontally and ensure float type
combined = np.column_stack(
    (DIM.astype(float), LOOM.astype(float), SWEEP.astype(float))
)

# Fit PCA (scikit-learn expects samples as rows and features as columns)
pca = PCA(n_components=5, svd_solver="full")
pca.fit(combined.T)

var_exp = pca.explained_variance_ratio_ * 100

# Project each separately to get individual trajectories
scores_DIM = pca.transform(DIM.T).T
scores_LOOM = pca.transform(LOOM.T).T
scores_SWEEP = pca.transform(SWEEP.T).T

n = scores_LOOM.shape[1]
for name, trajectory in {"Dimming": scores_DIM, "Looming": scores_LOOM, "Sweeping": scores_SWEEP}.items():
    if not np.isfinite(trajectory).all():
        raise ValueError(f"Non-finite PCA scores: {name}")


### Plot PCA Trajectories


In [ ]:
tab_cmap = sns.color_palette(palette='tab10', n_colors=10, desat=0.5)
vlag_cmap = sns.color_palette("vlag", n_colors=10, desat=1)


In [ ]:
# Shared three-dimensional PCA trajectory plot
savefig = True
show_ticks = False
show_legend = False

# Mark onset in the cropped PCA time coordinates.
event_markers = {"stim_start": (STIM_START - PCA_START, "o")}

# ------------------------------------------------------------------
# Data
# ------------------------------------------------------------------
trajectories = {
    "Dimming":  np.asarray(scores_DIM, dtype=float)[:3],
    "Looming":  np.asarray(scores_LOOM, dtype=float)[:3],
    "Sweeping": np.asarray(scores_SWEEP, dtype=float)[:3],
}

for name, trajectory in trajectories.items():
    if trajectory.ndim != 2 or trajectory.shape[0] != 3:
        raise ValueError(
            f"{name} must have shape (3, time); "
            f"received {trajectory.shape}"
        )

# ------------------------------------------------------------------
# Calculate equal limits before drawing projections
# ------------------------------------------------------------------
all_xyz = np.concatenate(list(trajectories.values()), axis=1)

# Use samples for which all three PCs are finite
finite_samples = np.all(np.isfinite(all_xyz), axis=0)
all_finite = all_xyz[:, finite_samples]

if all_finite.shape[1] == 0:
    raise ValueError("No finite three-dimensional PCA samples were found.")

data_min = all_finite.min(axis=1)
data_max = all_finite.max(axis=1)
data_mid = (data_min + data_max) / 2

# Same numerical span for PC1, PC2 and PC3
common_span = float(np.max(data_max - data_min))
if common_span <= 0:
    common_span = 1.0

# Equal cube plus 5% clearance on each side
half_span = 0.55 * common_span
limits = np.column_stack((
    data_mid - half_span,
    data_mid + half_span,
))

z_floor = limits[2, 0]
y_wall = limits[1, 1]
x_wall = limits[0, 0]

# ------------------------------------------------------------------
# Figure
# ------------------------------------------------------------------
sns.set_theme(
    style="white",
    context="paper",
    font=FONT_FAMILY,
)

colors = [tab_cmap[5], vlag_cmap[-1], tab_cmap[2]] #sns.color_palette("rocket", n_colors=3)


width_mm = 55
height_mm = 55

fig = plt.figure(
    figsize=(width_mm / 25.4, height_mm / 25.4),
    layout="constrained",
)

ax = fig.add_subplot(111, projection="3d")

# Freeze limits before adding projections so they cannot alter autoscaling
ax.set_xlim(*limits[0])
ax.set_ylim(*limits[1])
ax.set_zlim(*limits[2])

# Equal visual scaling in all three dimensions
ax.set_box_aspect((1, 1, 1))
#ax.view_init(elev=30, azim=15)
#ax.view_init(elev=30, azim=-45)


# ------------------------------------------------------------------
# Floor projections first
# ------------------------------------------------------------------
for (label, trajectory), color in zip(trajectories.items(), colors):
    x, y, z = trajectory

    ax.plot(
        x,
        y,
        np.full(x.shape, z_floor, dtype=float),
        color=color,
        alpha=0.15,
        linewidth=0.6,
        label="_nolegend_",
    )

# ------------------------------------------------------------------
# Wall projections second
# ------------------------------------------------------------------
for (label, trajectory), color in zip(trajectories.items(), colors):
    x, y, z = trajectory

    ax.plot(
        x,
        np.full(x.shape, y_wall, dtype=float),
        z,
        color=color,
        alpha=0.15,
        linewidth=0.6,
        label="_nolegend_",
    )

# ------------------------------------------------------------------
# Wall projections third
# ------------------------------------------------------------------
for (label, trajectory), color in zip(trajectories.items(), colors):
    x, y, z = trajectory

    ax.plot(
        np.full(x.shape, x_wall, dtype=float),
        y,
        z,
        color=color,
        alpha=0.15,
        linewidth=0.6,
        label="_nolegend_",
    )

# ------------------------------------------------------------------
# Main trajectories and temporal markers
# ------------------------------------------------------------------
for (label, trajectory), color in zip(trajectories.items(), colors):
    x, y, z = trajectory

    ax.plot(
        x,
        y,
        z,
        color=color,
        linewidth=1,
        label=label,
    )

    for index, marker in [*event_markers.values(), (STIM_END[label] - PCA_START, "^")]:
        if 0 <= index < trajectory.shape[1]:
            point = trajectory[:, index]

            if np.all(np.isfinite(point)):
                ax.scatter(
                    point[0],
                    point[1],
                    point[2],
                    color=[color],
                    marker=marker,
                    s=10,
                    depthshade=False,
                )

# ------------------------------------------------------------------
# Labels and formatting
# ------------------------------------------------------------------
ax.set_xlabel(
    f"PC1 ({var_exp[0]:.1f}%)",
    fontsize=8,
    labelpad=-8,
)
ax.set_ylabel(
    f"PC2 ({var_exp[1]:.1f}%)",
    fontsize=8,
    labelpad=-8,
)
ax.set_zlabel(
    f"PC3 ({var_exp[2]:.1f}%)",
    fontsize=8,
    labelpad=-8,
)

ax.grid(False)

if show_ticks:
    ax.tick_params(axis="x", labelsize=6, pad=-1)
    ax.tick_params(axis="y", labelsize=6, pad=-1)
    ax.tick_params(axis="z", labelsize=6, pad=-1)
else:
    ax.set(xticks=[], yticks=[], zticks=[])

if show_legend:
    # Inside the figure so the 3D plot is not compressed at 55 mm.
    ax.legend(
        loc="upper left",
        frameon=False,
        fontsize=6,
        handlelength=1.4,
        handletextpad=0.4,
        borderaxespad=0.2,
    )

# ------------------------------------------------------------------
# Export
# ------------------------------------------------------------------

if savefig:
    save_figure(fig, "PCA_Panel_B_v02.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    save_figure(fig, "PCA_Panel_B_v02.svg",
        bbox_inches="tight",pad_inches=.1,
        facecolor="white",
    )

plt.show()
plt.close("all")


# Panel C: Number of Areas vs Activity


In [ ]:
# ---------- Activity Counts & Fractions ---------- #
stimuli_p = {"looming": apLT, "dimming": apDT, "sweep": apSB}
activity_stats = {}
threshold = FDR_ALPHA

for name, p_val in stimuli_p.items():
    mask = p_val < threshold
    activity_stats[name] = {
        "n_active": np.count_nonzero(mask),
        "f_active": np.mean(mask)
    }

# ---------- Shared Significant Regions ---------- #
shared_mask = (apLT < threshold) & (apDT < threshold) & (apSB < threshold)
shared_idx = np.where(shared_mask)[0]
shared_labels = [region_labels[i] for i in shared_idx]

print(f"Number of shared active areas: {len(shared_idx)}")

# ---------- Shared Response Summary Table ---------- #
shared_response = pd.DataFrame({
    "region": shared_labels,
    "looming_mean": np.mean(aLT[shared_idx], axis=1),
    "dimming_mean": np.mean(aDT[shared_idx], axis=1),
    "sweep_mean":   np.mean(aSB[shared_idx], axis=1),
})

# Compute absolute values and sort by maximum absolute response
for prefix in ["looming", "dimming", "sweep"]:
    shared_response[f"{prefix}_abs"] = np.abs(shared_response[f"{prefix}_mean"])

shared_response["max_abs_response"] = shared_response[
    ["looming_abs", "dimming_abs", "sweep_abs"]
].max(axis=1)

shared_response = shared_response.sort_values("max_abs_response", ascending=False)
display(shared_response)  # Use display() or leave as shared_response in notebooks


# ---------- Long-Form DataFrame (RR) ---------- #
stimuli_matrices = [("looming", aLT), ("dimming", aDT), ("sweep", aSB)]
frames = []

for stimulus_name, response_matrix in stimuli_matrices:
    n_regions, n_animals = response_matrix.shape
    region_idx = np.repeat(np.arange(n_regions), n_animals)
    animal_idx = np.tile(np.arange(n_animals), n_regions)

    frames.append(pd.DataFrame({
        "region": [region_labels[i] for i in region_idx],
        "animal": animal_idx,
        "stimulus": stimulus_name,
        "response": response_matrix[region_idx, animal_idx],
    }))

RR = pd.concat(frames, ignore_index=True)
display(RR)


In [ ]:
Loom_nactive = np.count_nonzero(apLT < threshold)
Dim_nactive = np.count_nonzero(apDT < threshold)
Sweep_nactive = np.count_nonzero(apSB < threshold)

# ==========================================
# 1. PREPARE DABEST STATISTICAL ANALYSIS
# ==========================================
shared_region_idx = shared_idx
shared_region_labels = shared_labels

shared_mag_df = pd.DataFrame(
    {
        "Region": shared_region_labels,
        "Looming": np.mean(np.abs(aLT[shared_region_idx, :]), axis=1),
        "Dimming": np.mean(np.abs(aDT[shared_region_idx, :]), axis=1),
        "Sweep": np.mean(np.abs(aSB[shared_region_idx, :]), axis=1),
    }
)

# Reshape for DABEST: one row per region, one value per group
dabest_df = shared_mag_df.melt(
    id_vars="Region",
    var_name="Group",
    value_name="Magnitude",
).rename(columns={"Region": "id"})

if len(shared_mag_df) < 2:
    raise ValueError("Panel C needs at least two shared active regions")

my_data = dabest.load(
    dabest_df,
    x="Group",
    y="Magnitude",
    idx=("Looming", "Dimming", "Sweep"),
    paired="baseline",
    id_col="id",
    random_seed=RANDOM_SEED,
)

stats_df = my_data.cohens_d.results
diff_col = "difference"
differences = stats_df[diff_col].astype(float)


# ==========================================
# 2. FINAL DUAL-AXIS PLOT
# ==========================================
width_mm = 60  # Width in millimeters
height_mm = 50  # Height in millimeters
fig, ax1 = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4))

sns.set_theme(style="white")

conditions = ["Looming", "Dimming", "Sweep"]
active = [Loom_nactive, Dim_nactive, Sweep_nactive]
x = np.arange(len(conditions))
width = 0.6

# Cohen's d across shared regions, relative to Looming, with BCa intervals.
shared_mean_response = np.array([
    0.0, 
    differences[0],
    differences[1]
])
ci_low = np.r_[0.0, stats_df["bca_low"].to_numpy()]
ci_high = np.r_[0.0, stats_df["bca_high"].to_numpy()]

# Left axis: Number of active regions
ax1.bar(x, active, width=width, color="#979B9E", label="Active regions")
ax1.set_xticks(x)
ax1.set_xticklabels(conditions, fontsize=7, rotation=30, ha="right")
ax1.set_yticks([0, 50, 100])
ax1.set_yticklabels([0, 50, 100], fontsize=6)
ax1.set_ylabel("Number of active regions", fontsize=8)
ax1.set_ylim(0, max(active) * 1.1)

for i, act in enumerate(active):
    ax1.text(i, act + max(active) * 0.02, f"{act}", ha="center", va="bottom", fontsize=6)

# Right axis: Mean response size with error bars
ax2 = ax1.twinx()
ax2.errorbar(
    x,
    shared_mean_response,

    fmt="o-",
    color="#050505",
    linewidth=2,
    markersize=6,
    capsize=0,
    elinewidth=1,
    label="Cohen’s d relative to Looming; 95% BCa CI",
)
ax2.vlines(x, ci_low, ci_high, color="black", linewidth=1)
ax2.set_ylabel("Cohen’s d (vs Looming)", fontsize=8)
ax2.set_yticks([-2, -1, 0, 1])
ax2.set_yticklabels([-2, -1, 0, 1], fontsize=6)
ax2.set_ylim(-2, 1)


# Spine cleanup
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(True)
ax2.spines["right"].set_visible(True)

plt.tight_layout()

# Save graphics
save_figure(plt.gcf(), "Panel_C_activity_summary.png", dpi=300, bbox_inches="tight")
save_figure(plt.gcf(), "Panel_C_activity_summary.svg", bbox_inches="tight")

plt.show()
plt.close("all")


# Panel D - Example Responses


In [ ]:
ACA = np.arange(1, 3)
AUD = np.arange(4, 8)
RSP = np.arange(18, 21)
SSp = np.arange(21, 28)
VIS = np.arange(32, 45)
COA = np.arange(50, 53)
BLA = np.arange(54, 57)
BMA = np.arange(57, 59)
CEA = np.arange(59, 62)
HIPP = np.arange(66, 78)
SUBp = np.arange(80, 84)
CPc = np.arange(87, 92)
CPm = np.arange(92, 101)
GP = np.arange(102, 104)
LS = np.arange(104, 107)
LG = np.arange(128, 131)
MG = np.arange(133, 136)
PTR = np.arange(195, 200)
MLR = np.arange(201, 203)
IC = np.arange(206, 209)
MRN = np.arange(212, 216)
PAG = np.arange(221, 228)
SC = np.arange(236, 246)
SN = np.arange(246, 248)


In [ ]:
# Pathways
# --> SCs, LP, CPc, LA, VISpor (putative WF pathways)
SCs = np.arange(242, 246)
LP = 131
CPc = np.arange(87, 90)
VISpor = 43

# --> SCs, SCi, PAGd, CUN (putative NF pathway)
SCs = np.arange(242, 246)
SCi = np.arange(238, 243)
PAGd = np.array([222, 225])
MRN = np.array([202, 212, 213, 214, 215])

# --> LGd, VISp(pl), VISl, VISli, VISpl, VISpor, TEa (Ventral stream "what")
LGd = 128
VISv = np.array([39, 35, 36, 41, 43, 29])

# --> LGd, VISp(al), VISp(am), VISp(pm), VISal, VISam, VISpm, VISrl, RSPagl, RSPd, APr, LD, POST, CA1d, CA3d (dorsal stream "where")
LGd = 128
VISd = np.array([37, 38, 40, 33, 34, 42, 44, 18, 19, 9, 127, 81, 66, 72])


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ---------- Define Pathways & Areas ---------- #
pathways = {
    "WF Pathway": {
        "SCs": np.arange(242, 246),
        "LP": np.array([131]),
        "CPc": np.arange(87, 90),
        "Amg": np.arange(55,58)
    },
    "NF Pathway": {
        "__empty_left__": None,          
        "SCd": np.arange(236, 243),       
        "PAGd": np.arange(222, 226),     
        #"MRN": np.arange(212, 216)
        "MLR": np.arange(201, 203)
    },
    "Cortical Visual Stream": {
        "LGd": np.array([128]),
        "VISv": np.array([39, 35, 36, 41, 43, 29]),
        "VISd": np.array([37, 38, 40, 33, 34, 42, 44, 18, 19, 9, 127, 81, 66, 72]),
        "ACA": np.arange(1, 3)
    }
}

stimuli_data = {
    "Looming": LoomL_smooth,
    "Dimming": DimL_smooth,
    "Sweeping": SweepL_smooth
}
#colors = [tab_cmap[5], vlag_cmap[-1], tab_cmap[2]] 
stim_colors = {
    "Looming": vlag_cmap[-1],      # Red
    "Dimming": tab_cmap[5],      # Blue
    "Sweeping": tab_cmap[2] # Green
}


stim_ends = {
    "Looming": 41,
    "Dimming": 41,
    "Sweeping": 46
}

# ---------- Setup Figure (95 mm x 52 mm) ---------- #
width_mm = 95
height_mm = 52
fig = plt.figure(figsize=(width_mm / 25.4, height_mm / 25.4))

max_cols = max(len(p) for p in pathways.values())
sns.set_theme(font=FONT_FAMILY, style="ticks")

gs = fig.add_gridspec(len(pathways), max_cols, hspace=0.4, wspace=0.3)

baseline_start, baseline_end = BASELINE.start, BASELINE.stop

for row_idx, (pathway_name, regions_dict) in enumerate(pathways.items()):
    for col_idx, (region_name, indices) in enumerate(regions_dict.items()):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        
        if region_name.startswith("__empty__") or indices is None:
            ax.axis('off')
            continue
            
        valid_idx = np.setdiff1d(indices, NoplotIndexes)
        
        if len(valid_idx) > 0:
            local_min, local_max = float('inf'), float('-inf')
            
            for stim_name, data in stimuli_data.items():
                raw_response = np.mean(data[valid_idx, :], axis=0)
                baseline = np.mean(raw_response[baseline_start:baseline_end])
                norm_response = raw_response - baseline
                
                ax.plot(norm_response, color=stim_colors[stim_name], linewidth=1, label=stim_name)
                
                local_min = min(local_min, np.min(norm_response[20:70]))
                local_max = max(local_max, np.max(norm_response[20:70]))
                    
            local_range = local_max - local_min
            if local_range == 0: 
                local_range = 0.1
            ax.set_ylim(local_min - local_range * 0.1, local_max + local_range * 0.1)
                    
            ax.axvline(STIM_START, color=".4", linewidth=0.7)
            for stim, end_time in stim_ends.items():
                ax.axvline(end_time, color=stim_colors[stim], linewidth=0.7, linestyle="--", alpha=0.5)
                
            ax.set_title(region_name, fontweight="bold", fontsize=8, loc='left', pad=3)
            
        ax.set_xlim(20, 70)
        # Remove all axes, ticks, labels, and spines completely
        ax.axis('off')
        

fig.legend(handles=[plt.Line2D([0], [0], color=c, label=n) for n, c in stim_colors.items()], loc="upper center", bbox_to_anchor=(0.5, 1.08), ncol=3, frameon=False, fontsize=6)

# Calibrated 2-second bar in sample coordinates on the final axes.
ax.set_xlim(20, 70)
ax.plot([60, 60 + 2 / SAMPLE_INTERVAL_S], [-0.12, -0.12],
        transform=ax.get_xaxis_transform(), color="black", clip_on=False)
ax.text(60 + 1 / SAMPLE_INTERVAL_S, -0.22, "2 s", ha="center",
        transform=ax.get_xaxis_transform(), fontsize=6)

if savefig == 1:
    save_figure(plt.gcf(), "Panel_D_v02.png", dpi=300, bbox_inches="tight")
    save_figure(plt.gcf(), "Panel_D_v02.svg", bbox_inches="tight")

plt.show()
plt.close("all")


# Brain Maps of Activity


### Map Responses onto Atlas


In [ ]:
response_values = {
    "Looming": res_LT['effect_size'],
    "Dimming": res_DT['effect_size'],
    "Sweeping": res_SB['effect_size'],
}

region_values = {
    "Looming": res_LT['effect_size'],
    "Dimming": res_DT['effect_size'],
    "Sweeping": res_SB['effect_size'],
}

# p and q values
p_values = {
    "Looming": pLT,
    "Dimming": pDT,
    "Sweeping": pSB
}

q_values = {
    "Looming": apLT,
    "Dimming": apDT,
    "Sweeping": apSB
}
for stimulus, values in region_values.items():
    print(stimulus, values.shape)


In [ ]:
ORIENTATION = "coronal"

slice_specs = [(f"Coronal {f:.0%}", "fraction of atlas axis", f)
               for f in [0.32, 0.45, 0.50, 0.55, 0.65, 0.80]]

n_coronal_planes = atlas_regions.shape[1]
plane_numbers = [
    int(round(1 + fraction * (n_coronal_planes - 1)))
    for _, _, fraction in slice_specs
]
display(
    pd.DataFrame(
        {
            "label": [name for name, _, _ in slice_specs],
            "target": [target for _, target, _ in slice_specs],
            "plane": plane_numbers,
        }
    )
)

values_for_plot = {}
for stimulus, values in region_values.items():
    current = values.copy()
    if SIGNIFICANT_ONLY:
        current[~(q_values[stimulus] < FDR_ALPHA)] = np.nan
    values_for_plot[stimulus] = current

all_plot_values = np.concatenate(list(values_for_plot.values()))
finite_values = all_plot_values[np.isfinite(all_plot_values)]
color_limit = np.max(np.abs(finite_values)) if finite_values.size else 1.0
if color_limit == 0:
    color_limit = 1.0

cmap = sns.diverging_palette(220, 20, sep=30, as_cmap=True).with_extremes(
    bad=(1.0, 1.0, 1.0, 0.0)
)
selected_atlas_labels = np.unique(np.concatenate(selection.parts))
print(f"Symmetric color range: {-color_limit:.4g} to {color_limit:.4g}")


In [ ]:
width_mm = 200
height_mm = 60
stimuli = ["Looming", "Dimming", "Sweeping"]
fig, axes = plt.subplots(
    len(stimuli),
    len(slice_specs),
    figsize=(width_mm / 25.4, height_mm / 25.4),
    facecolor="white",
    constrained_layout=True,
)

for column, ((title, target, _), plane_number) in enumerate(
    zip(slice_specs, plane_numbers, strict=True)
):
    label_slice = atlas_plane(atlas_regions, ORIENTATION, plane_number)
    tissue_mask = label_slice > 0
    selected_mask = np.isin(label_slice, selected_atlas_labels)

    for row, stimulus in enumerate(stimuli):
        ax = axes[row, column]
        ax.set_facecolor("white")

        # Use the translated MATLAB recoloring function on this 2-D label slice.
        activity_slice = recolor_map_double(
            {"Regions": label_slice}, selection, values_for_plot[stimulus]
        )
        activity_masked = np.ma.masked_where(
            ~selected_mask | ~np.isfinite(activity_slice), activity_slice
        )
        silhouette = np.ma.masked_where(~tissue_mask, np.ones_like(label_slice, dtype=float))

        ax.imshow(silhouette, cmap="Greys", vmin=0, vmax=1.2, alpha=0.12)
        ax.imshow(
            activity_masked,
            cmap=cmap,
            vmin=-color_limit,
            vmax=color_limit,
            interpolation="nearest",
        )
        draw_borders(
            atlas, ORIENTATION, plane_number, ax=ax, color="white", linewidth=0.15
        )
        ax.set_axis_off()

        if row == 0:
            ax.set_title(f"{title}\nPlane {plane_number}", fontsize=7, fontweight="bold")
        if column == 0:
            ax.text(
                -0.08,
                0.5,
                stimulus,
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="right",
                fontsize=10,
                fontweight="bold",
            )

mappable = ScalarMappable(norm=Normalize(-color_limit, color_limit), cmap=cmap)

cbar = fig.colorbar(
    mappable,
    ax=axes.ravel().tolist(),
    fraction=0.02,
    pad=0.05,
    shrink=.5
)

cbar.set_ticks([-color_limit, 0, color_limit])
cbar.ax.tick_params(labelsize=6)
cbar.set_label("Cohen’s d", fontsize=8, weight="normal")
cbar.outline.set_visible(False)

# Save as PNG (high resolution)
save_figure(fig, "Panel_E_coronal.png", dpi=300, bbox_inches="tight")

# Save as SVG (vector format, good for Illustrator/Inkscape)
save_figure(fig, "Panel_E_coronal.svg", bbox_inches="tight")

plt.show()
plt.close("all")


### Sagittal


In [ ]:
ORIENTATION = "sagittal"

sagittal_fractions = [0.15, 0.25, 0.3, 0.4]

n_sagittal_planes = atlas_regions.shape[2]

plane_numbers = [
    int(round(1 + fraction * (n_sagittal_planes - 1)))
    for fraction in sagittal_fractions
]

slice_specs = [(f"Sagittal {f:.0%}", "fraction of atlas axis", f)
               for f in sagittal_fractions]

print("Sagittal plane numbers:", plane_numbers)


In [ ]:
# Exclude the atlas root/background label.
root_index = selection.acr.index("root")
root_labels = selection.parts[root_index]

selected_labels_without_root = np.setdiff1d(
    selected_atlas_labels,
    root_labels,
)

# Make masked pixels transparent so the white background is visible.
activity_cmap = cmap.with_extremes(
    bad=(1.0, 1.0, 1.0, 0.0)
)

silhouette_cmap = plt.colormaps["Greys"].with_extremes(
    bad=(1.0, 1.0, 1.0, 0.0)
)

stimuli = ["Looming", "Dimming", "Sweeping"]
width_mm = 200
height_mm = 60
fig, axes = plt.subplots(
    len(stimuli),
    len(slice_specs),
    figsize=(width_mm / 25.4, height_mm / 25.4),
    facecolor="white",
    constrained_layout=True,
)

fig.patch.set_facecolor("white")

for column, ((title, target, _), plane_number) in enumerate(
    zip(slice_specs, plane_numbers, strict=True)
):
    label_slice = atlas_plane(
        atlas_regions,
        ORIENTATION,
        plane_number,
    )

    # The atlas exterior may be label 0 or the root label.
    outside_mask = (
        (label_slice == 0)
        | np.isin(label_slice, root_labels)
    )

    tissue_mask = ~outside_mask

    selected_mask = (
        tissue_mask
        & np.isin(label_slice, selected_labels_without_root)
    )

    silhouette = np.ma.array(
        np.ones_like(label_slice, dtype=float),
        mask=~tissue_mask,
    )

    for row, stimulus in enumerate(stimuli):
        ax = axes[row, column]
        ax.set_facecolor("white")

        activity_slice = recolor_map_double(
            {"Regions": label_slice},
            selection,
            values_for_plot[stimulus],
        )

        activity_mask = (
            outside_mask
            | ~selected_mask
            | ~np.isfinite(activity_slice)
        )

        activity_masked = np.ma.array(
            activity_slice,
            mask=activity_mask,
        )

        # Light-gray brain silhouette
        ax.imshow(
            silhouette,
            cmap=silhouette_cmap,
            vmin=0,
            vmax=1.2,
            alpha=0.12,
            interpolation="nearest",
        )

        # Activity overlay
        ax.imshow(
            activity_masked,
            cmap=activity_cmap,
            vmin=-color_limit,
            vmax=color_limit,
            interpolation="bilinear",
        )

        draw_borders(
            atlas,
            ORIENTATION,
            plane_number,
            ax=ax,
            color="white",
            linewidth=0,
        )

        ax.set_axis_off()

        if row == 0:
            ax.set_title(
                f"{title}\nPlane {plane_number}",
                fontsize=7,
                fontweight="bold",
            )

        if column == 0:
            ax.text(
                -0.08,
                0.5,
                stimulus,
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="right",
                fontsize=10,
                fontweight="bold",
            )

mappable = ScalarMappable(norm=Normalize(-color_limit, color_limit), cmap=cmap)

cbar = fig.colorbar(
    mappable,
    ax=axes.ravel().tolist(),
    fraction=0.02,
    pad=0.05,
    shrink=.5
)

cbar.set_ticks([-color_limit, 0, color_limit])
cbar.ax.tick_params(labelsize=6)
cbar.set_label("Cohen’s d", fontsize=8, weight="normal")
cbar.outline.set_visible(False)

# Save as PNG (high resolution)
save_figure(fig, "Panel_E_sagittal.png", dpi=300, bbox_inches="tight")

# Save as SVG (vector format, good for Illustrator/Inkscape)
save_figure(fig, "Panel_E_sagittal.svg", bbox_inches="tight")


plt.show()
plt.close("all")


In [ ]:
# ---------- Sagittal activity maps with group averages and all regions ---------- #
# Keep the same 2x2 figure structure and four slices as the preceding sagittal map.
groups_to_plot = ["SCs", "SCi", "SCd", "PAGd", "PAGv", "LGv", "LGd", "LP"]
group_patterns = {
    "SCs": ("SCs",),
    "SCi": ("SCi",),
    "SCd": ("SCd",),
    "PAGd": ("PAGmd", "PAGpd"),
    "PAGv": ("PAGmv", "PAGpv"),
    "LGv": ("LGv",),
    "LGd": ("LGd",),
    "LP": ("LP",),
}

region_labels_array = np.asarray(region_labels, dtype=object)
region_labels_lower = np.char.lower(region_labels_array.astype(str))
group_indices = {}
for group_name in groups_to_plot:
    patterns = group_patterns[group_name]
    if group_name == "LP":
        matches = np.flatnonzero(region_labels_lower == "lp")
    else:
        matches = np.flatnonzero(
            np.logical_or.reduce(
                [np.char.startswith(region_labels_lower, pattern.lower()) for pattern in patterns]
            )
        )
    if matches.size == 0:
        raise ValueError(f"Requested group was not found: {group_name}")
    group_indices[group_name] = matches

group_atlas_labels = {
    group_name: np.unique(
        np.concatenate([np.asarray(selection.parts[index]) for index in indices])
    )
    for group_name, indices in group_indices.items()
}
selected_group_atlas_labels = np.unique(np.concatenate(list(group_atlas_labels.values())))

print("Groups averaged on each map:")
for group_name, indices in group_indices.items():
    print(f"{group_name}: {region_labels_array[indices].tolist()}")

activity_cmap = cmap.with_extremes(bad=(1.0, 1.0, 1.0, 0.0))
silhouette_cmap = plt.colormaps["Greys"].with_extremes(bad=(1.0, 1.0, 1.0, 0.0))

for stimulus in stimuli:
    # Start with the activity region values, then replace requested groups by their mean.
    group_average_values = np.asarray(values_for_plot[stimulus], dtype=float).copy()
    for group_name in groups_to_plot:
        group_average_values[group_indices[group_name]] = np.nanmean(
            values_for_plot[stimulus][group_indices[group_name]]
        )

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(165 / 25.4, 150 / 25.4),
        facecolor="white",
        constrained_layout=True,
    )
    axes = axes.ravel()
    fig.patch.set_facecolor("white")

    for ax, (slice_spec, plane_number) in zip(
        axes,
        zip(slice_specs, plane_numbers, strict=True),
        strict=True,
    ):
        title, target, _ = slice_spec
        label_slice = atlas_plane(atlas_regions, ORIENTATION, plane_number)
        outside_mask = (label_slice == 0) | np.isin(label_slice, root_labels)
        tissue_mask = ~outside_mask
        selected_mask = tissue_mask & np.isin(label_slice, selected_atlas_labels)
        silhouette = np.ma.array(np.ones_like(label_slice, dtype=float), mask=~tissue_mask)

        activity_slice = recolor_map_double(
            {"Regions": label_slice},
            selection,
            group_average_values,
        )
        activity_masked = np.ma.array(
            activity_slice,
            mask=outside_mask | ~selected_mask | ~np.isfinite(activity_slice),
        )

        ax.imshow(
            silhouette,
            cmap=silhouette_cmap,
            vmin=0,
            vmax=1.2,
            alpha=0.12,
            interpolation="nearest",
        )
        ax.imshow(
            activity_masked,
            cmap=activity_cmap,
            vmin=-color_limit,
            vmax=color_limit,
            interpolation="nearest",
        )
        draw_borders(
            atlas,
            ORIENTATION,
            plane_number,
            ax=ax,
            color="white",
            linewidth=0,
        )
        ax.set_title(f"{title}\n{target}", fontsize=7, fontweight="bold", pad=2)
        ax.set_axis_off()

    fig.suptitle(
        f"{stimulus} group-average activity with all regions ({ORIENTATION})",
        fontsize=9,
        fontweight="bold",
    )
    mappable = ScalarMappable(
        norm=Normalize(-color_limit, color_limit),
        cmap=activity_cmap,
    )
    mappable.set_array([])
    cbar = fig.colorbar(
        mappable,
        ax=axes.tolist(),
        orientation="horizontal",
        fraction=0.035,
        pad=0.02,
        shrink=0.7,
    )
    cbar.set_ticks([-color_limit, 0, color_limit])
    cbar.ax.tick_params(labelsize=6)
    cbar.set_label("Mean Cohen’s d; individual regions elsewhere", fontsize=7)
    cbar.outline.set_visible(False)

    if globals().get("savefig", True):
        filename = stimulus.lower()
        save_figure(fig, f"Panel_E_group_average_all_regions_{ORIENTATION}_{filename}.png",
            dpi=300,
            bbox_inches="tight",
            facecolor="white",
        )
        save_figure(fig, f"Panel_E_group_average_all_regions_{ORIENTATION}_{filename}.svg",
            bbox_inches="tight",
            facecolor="white",
        )

    plt.show()
    plt.close("all")


# Panel F — paired CNO versus Vehicle

Animals are already matched in the input arrays (confirmed by the author). Differences are **CNO minus Vehicle**. As in the source, both one-sided tests are corrected separately and their significant-region sets are combined for the printed count. This union is **not** a two-sided test with combined FDR control. Downstream heatmaps/maps use the `less` tail. Region selection uses the Vehicle activity results and is not independent of the comparison. These choices are retained for reproduction, not endorsed as a new confirmatory analysis.


In [ ]:
# Preserve the source rule: exclude regions with q > .05 for any stimulus,
# plus the non-plotted regions. This is selection using the same Vehicle data.
indices2exclude = np.unique(np.r_[
    np.flatnonzero((apLT > FDR_ALPHA) | (apDT > FDR_ALPHA) | (apSB > FDR_ALPHA)),
    NoplotIndexes,
])


In [ ]:
# Calculate difference (e.g., CNO minus Vehicle) per animal
# Note: If your data is paired by animal, subtraction is done directly along the animal axis.
diff_aLT =  aLTcno - aLTveh
diff_aDT = aDTcno - aDTveh
diff_aSB = aSBcno - aSBveh


In [ ]:
# Store each tail explicitly; downstream heatmaps use the original final tail (less).
cno_by_tail = {}
for alternative in ("greater", "less"):
    cno_by_tail[alternative] = {
        stimulus: run_estimation_and_permutation_test(
            data, effect_type="cohens_d", alpha=FDR_ALPHA,
            n_resamples=N_PERMUTATIONS, n_bootstraps=N_BOOTSTRAPS_CNO,
            no_plot_indexes=indices2exclude, alternative=alternative,
            random_state=RANDOM_SEED,
        )
        for stimulus, data in {"LT": diff_aLT, "DT": diff_aDT, "SB": diff_aSB}.items()
    }
res_CNO_LT, res_CNO_DT, res_CNO_SB = [cno_by_tail["less"][s] for s in ("LT", "DT", "SB")]
for stimulus in ("LT", "DT", "SB"):
    combined = np.union1d(cno_by_tail["less"][stimulus]["sig_rows"],
                          cno_by_tail["greater"][stimulus]["sig_rows"])
    print(f"{stimulus}: {len(combined)} regions in union of separately corrected tails")


In [ ]:
# ==========================================
# 0. ALIGN VARIABLES FROM PREVIOUS PIPELINE
# ==========================================
LoomV_smooth = smooth_trace(np.squeeze(np.mean(LoomT_V, axis=2)))
LoomC_smooth = smooth_trace(np.squeeze(np.mean(LoomT_C, axis=2)))

DimV_smooth  = smooth_trace(np.squeeze(np.mean(DimmT_V, axis=2)))
DimC_smooth  = smooth_trace(np.squeeze(np.mean(DimmT_C, axis=2)))

SweepV_smooth = smooth_trace(np.squeeze(np.mean(SweepT_V, axis=2)))
SweepC_smooth = smooth_trace(np.squeeze(np.mean(SweepT_C, axis=2)))

# Pulling p-values and effect sizes from results dictionaries
apLT_VcC = res_CNO_LT['p_corrected']
apDT_VcC = res_CNO_DT['p_corrected']
apST_VcC = res_CNO_SB['p_corrected']

eff_LT = res_CNO_LT['effect_size']
ci_low_LT = res_CNO_LT['ci_lower']
ci_high_LT = res_CNO_LT['ci_upper']

eff_DT = res_CNO_DT['effect_size']
ci_low_DT = res_CNO_DT['ci_lower']
ci_high_DT = res_CNO_DT['ci_upper']

eff_SB = res_CNO_SB['effect_size']
ci_low_SB = res_CNO_SB['ci_lower']
ci_high_SB = res_CNO_SB['ci_upper']

# ==========================================
# 1. ORGANIZE AND FILTER DATA
# ==========================================
keep_idx = np.setdiff1d(np.arange(LoomV_smooth.shape[0]), NoplotIndexes)

Loom_V, Loom_C, Loom_Diff = LoomV_smooth[keep_idx], LoomC_smooth[keep_idx], (LoomC_smooth - LoomV_smooth)[keep_idx]
Dimm_V, Dimm_C, Dimm_Diff = DimV_smooth[keep_idx], DimC_smooth[keep_idx], (DimC_smooth - DimV_smooth)[keep_idx]
Sweep_V, Sweep_C, Sweep_Diff = SweepV_smooth[keep_idx], SweepC_smooth[keep_idx], (SweepC_smooth - SweepV_smooth)[keep_idx]

apLT_plot, apDT_plot, apSB_plot = apLT_VcC[keep_idx], apDT_VcC[keep_idx], apST_VcC[keep_idx]

eff_LT_plot, ci_low_LT_plot, ci_high_LT_plot = eff_LT[keep_idx], ci_low_LT[keep_idx], ci_high_LT[keep_idx]
eff_DT_plot, ci_low_DT_plot, ci_high_DT_plot = eff_DT[keep_idx], ci_low_DT[keep_idx], ci_high_DT[keep_idx]
eff_SB_plot, ci_low_SB_plot, ci_high_SB_plot = eff_SB[keep_idx], ci_low_SB[keep_idx], ci_high_SB[keep_idx]

# ==========================================
# 2. BRAIN REGIONS COLOR BARS & LABELS
# ==========================================
region_color_map = {
    "CTX": "#70F470", "CTXs": "#40E0D0", "TH": "#FF8080",
    "HY": "#E60000", "MB": "#FF70F7", "HBC": "#FF9900",
}

region_defs = [
    (CTXpl, region_color_map["CTX"]), (CTXspl, region_color_map["CTXs"]),
    (TH, region_color_map["TH"]), (HY, region_color_map["HY"]),
    (MB, region_color_map["MB"]), (HB_C, region_color_map["HBC"]),
]

region_segments = []
for orig_indices, color in region_defs:
    mask = np.isin(orig_indices, keep_idx)
    filtered_indices = np.where(mask)[0]
    if len(filtered_indices) > 0:
        y_start = np.searchsorted(keep_idx, orig_indices[filtered_indices[0]])
        y_end = np.searchsorted(keep_idx, orig_indices[filtered_indices[-1]]) + 1
        region_segments.append((y_start, y_end, color))

grouped_areas = {
    "ACA": np.arange(1, 3), "AUD": np.arange(4, 8), "RSP": np.arange(18, 21),
    "SSp": np.arange(21, 28), "VIS": np.arange(32, 45), "BLA": np.arange(54, 57), 
    "CEA": np.arange(59, 62), "HIPP": np.arange(66, 78), "SUBp": np.arange(80, 84), 
    "CPc": np.arange(87, 92), "CPm": np.arange(92, 101), "GP": np.arange(102, 104),
    "LG": np.arange(128, 131), "MG": np.arange(133, 136), "PTR": np.arange(195, 200),
    "MLR": np.arange(201, 203), "MRN": np.arange(212, 216), "PAG": np.arange(221, 228), 
    "SC": np.arange(236, 246), "SN": np.arange(246, 248)
}

threshold = FDR_ALPHA
group_extents_loom, group_extents_dim, group_extents_sweep = [], [], []

for label, orig_indices in grouped_areas.items():
    mask = np.isin(orig_indices, keep_idx)
    filtered_indices = np.where(mask)[0]

    if len(filtered_indices) > 0:
        y_positions = np.searchsorted(keep_idx, orig_indices[filtered_indices])
        y_min, y_max = y_positions[0] - 0.5, y_positions[-1] + 0.5
        y_center = np.mean([y_min, y_max])

        if np.mean(apLT_plot[y_positions] < threshold) > 0.1:
            group_extents_loom.append((y_min, y_max, y_center, label))
        if np.mean(apDT_plot[y_positions] < threshold) > 0.1:
            group_extents_dim.append((y_min, y_max, y_center, label))
        if np.mean(apSB_plot[y_positions] < threshold) > 0.1:
            group_extents_sweep.append((y_min, y_max, y_center, label))

# ==========================================
# 3. DEFINE CONFIGURATIONS FOR LOOPING
# ==========================================
stimuli_configs = [
    {
        "name": "Loom", "V": Loom_V, "C": Loom_C, "Diff": Loom_Diff,
        "eff": eff_LT_plot, "ci_low": ci_low_LT_plot, "ci_high": ci_high_LT_plot,
        "p_vals": apLT_plot, "extents": group_extents_loom
    },
    {
        "name": "Dim", "V": Dimm_V, "C": Dimm_C, "Diff": Dimm_Diff,
        "eff": eff_DT_plot, "ci_low": ci_low_DT_plot, "ci_high": ci_high_DT_plot,
        "p_vals": apDT_plot, "extents": group_extents_dim
    },
    {
        "name": "Sweep", "V": Sweep_V, "C": Sweep_C, "Diff": Sweep_Diff,
        "eff": eff_SB_plot, "ci_low": ci_low_SB_plot, "ci_high": ci_high_SB_plot,
        "p_vals": apSB_plot, "extents": group_extents_sweep
    }
]

# ==========================================
# 4. PLOT, FORMAT, AND SAVE EACH FIGURE
# ==========================================
sns.set_theme(font=FONT_FAMILY, style="white")
c_map = sns.diverging_palette(220, 20, sep=30, as_cmap=True)
heatmap_kwargs = dict(cmap=c_map, vmin=-0.01, vmax=0.01, aspect="auto", origin="upper")

# Adjusted width for a single stimulus figure (roughly 1/3 of your original wide plot)
width_mm = 130
height_mm = 150
y_limits_exact = (Loom_V.shape[0] - 0.5, -0.5)
y_vals = np.arange(Loom_V.shape[0])

savefig = 1

for stim in stimuli_configs:
    fig = plt.figure(figsize=(width_mm / 25.4, height_mm / 25.4))
    
    # 4 columns for data, 1 row for plot, 1 small row for colorbar
    gs = fig.add_gridspec(2, 4, height_ratios=[1, 0.08], hspace=0.3, wspace=0.25)
    
    ax_V = fig.add_subplot(gs[0, 0])
    ax_C = fig.add_subplot(gs[0, 1])
    ax_D = fig.add_subplot(gs[0, 2])
    ax_p = fig.add_subplot(gs[0, 3])

    # Plot Heatmaps
    ax_V.imshow(stim["V"], **heatmap_kwargs)
    ax_C.imshow(stim["C"], **heatmap_kwargs)
    hm1 = ax_D.imshow(stim["Diff"], **heatmap_kwargs)
    
    ax_V.set_title(f"{stim['name']} V", fontsize=8, fontweight="bold")
    ax_C.set_title(f"{stim['name']} C", fontsize=8, fontweight="bold")
    ax_D.set_title("Diff (C-V)", fontsize=8, fontweight="bold")

    # Add Brain Region Color Blocks to Vehicle Column
    for y0, y1, color in region_segments:
        ax_V.axhspan(y0 - 0.5, y1 - 0.5, xmin=-0.05, xmax=0.0, facecolor=color, edgecolor="none", zorder=20, clip_on=False)

    # Format Heatmap Axes
    for ax in [ax_V, ax_C, ax_D]:
        ax.axvline(STIM_START, color=".4", linewidth=0.8, label="StimStart")
        ax.axvline(46 if stim["name"] == "Sweep" else 41, color=".4", linewidth=0.8, linestyle="--", label="StimEnd")
        ax.set_xticks([20, 30, 60])
        ax.set_xticklabels([f"{(frame - STIM_START) * SAMPLE_INTERVAL_S:g}" for frame in [20, 30, 60]], fontsize=5)
        ax.set_xlabel("Time (s)", fontsize=7)
        ax.spines["top"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_yticks([])
        ax.set_ylim(y_limits_exact)
        ax.set_xlim(20, 70)
        ax.tick_params(axis="both", labelsize=5)

    # Plot Effect Sizes & CIs
    for y, e, l, h, p in zip(y_vals, stim["eff"], stim["ci_low"], stim["ci_high"], stim["p_vals"]):
        color = "indianred" if p < threshold else "black"
        alpha = 0.8 if p < threshold else 0.15
        ax_p.plot([l, h], [y, y], color=color, alpha=alpha, linewidth=0.5)
        ax_p.scatter([e], [y], color=color, alpha=alpha, s=2)

    # Format Effect Size Axes
    ax_p.axvline(0, color=".4", linewidth=0.8, linestyle=":")
    ax_p.spines["top"].set_visible(False)
    ax_p.spines["left"].set_visible(False)
    ax_p.spines["right"].set_visible(False)
    ax_p.set_ylim(y_limits_exact)
    ax_p.set_xlim(-2, 2)
    ax_p.set_yticks([])
    ax_p.set_xlabel("Cohen’s $d$", fontsize=6)
    ax_p.tick_params(axis="x", labelsize=5)

    # Group extent annotations (Right side labels)
    x_min_lim, x_max_lim = ax_p.get_xlim()
    for y_min, y_max, y_center, label in stim["extents"]:
        ax_p.vlines(x=x_max_lim * 1.1, ymin=y_min, ymax=y_max, color="gray", linewidth=1.2, transform=ax_p.transData, clip_on=False)
        ax_p.text(x=x_max_lim * 1.3, y=y_center, s=label, color="black", fontsize=5, va="center", ha="left", transform=ax_p.transData, clip_on=False)

    # Shrink effect size axis width slightly to make room for labels
    pos = ax_p.get_position()
    ax_p.set_position([pos.x0 - 0.005, pos.y0, pos.width * 0.45, pos.height])

    # Add Colorbar (centered below Vehicle and Control columns)
    cax = fig.add_subplot(gs[1, 0:2])
    cbar = fig.colorbar(hm1, cax=cax, orientation="horizontal")
    cbar.set_label("Activity / Difference", fontsize=7)
    cbar.ax.tick_params(labelsize=5)
    
    pos_cbar = cbar.ax.get_position()
    cbar.ax.set_position([pos_cbar.x0, pos_cbar.y0 + 0.03, pos_cbar.width, pos_cbar.height * 0.2])
    cbar.outline.set_visible(False)

    # Save out the individual figure
    if savefig == 1:
        save_figure(plt.gcf(), f"heatmap_{stim['name']}_V_vs_C_effect_sizes.png", dpi=300, bbox_inches="tight")
        save_figure(plt.gcf(), f"heatmap_{stim['name']}_V_vs_C_effect_sizes.svg", bbox_inches="tight")
    
    plt.show()
    plt.close("all")


# PCA - CNO vs VEH


### Calculate Trajectories


In [ ]:
# ---------- PCA using only keep_idx rows ---------- #

NoplotIndexes = np.array([
    1, 9, 65, 111, 114, 116, 161, 191, 142,
    251, 265, 297, 311, 312, 313, 314, 315,
    316, 317, 318, 319
]) - 1

n_regions = DimL.shape[0]
keep_idx = np.setdiff1d(np.arange(n_regions), NoplotIndexes)

pca_data = {
    "Looming VEH": smooth_trace(np.mean(LoomT_V, axis=2), sigma=PCA_SIGMA),
    "Looming CNO": smooth_trace(np.mean(LoomT_C, axis=2), sigma=PCA_SIGMA),
    "Dimming VEH": smooth_trace(np.mean(DimmT_V, axis=2), sigma=PCA_SIGMA),
    "Dimming CNO": smooth_trace(np.mean(DimmT_C, axis=2), sigma=PCA_SIGMA),
    "Sweeping VEH": smooth_trace(np.mean(SweepT_V, axis=2), sigma=PCA_SIGMA),
    "Sweeping CNO": smooth_trace(np.mean(SweepT_C, axis=2), sigma=PCA_SIGMA),
}

# Keep only non-excluded regions and time points from index 14 onward.
pca_data = {
    name: data[keep_idx, PCA_START:].astype(float)
    for name, data in pca_data.items()
}

trajectory_shapes = {data.shape for data in pca_data.values()}
if len(trajectory_shapes) != 1:
    raise ValueError(f"Trajectory shapes do not match: {trajectory_shapes}")

# One shared PCA space for all six VEH/CNO trajectories.
pca_training_data = np.vstack([
    trajectory.T for trajectory in pca_data.values()
])

pca = PCA(n_components=5, svd_solver="full")
pca.fit(pca_training_data)

var_exp = pca.explained_variance_ratio_ * 100

scores = {
    name: pca.transform(trajectory.T).T
    for name, trajectory in pca_data.items()
}

scores_loom_V = scores["Looming VEH"]
scores_loom_C = scores["Looming CNO"]
scores_dim_V = scores["Dimming VEH"]
scores_dim_C = scores["Dimming CNO"]
scores_sweep_V = scores["Sweeping VEH"]
scores_sweep_C = scores["Sweeping CNO"]

trajectories = {
    "Looming VEH": scores_loom_V[:3],
    "Looming CNO": scores_loom_C[:3],
    "Dimming VEH": scores_dim_V[:3],
    "Dimming CNO": scores_dim_C[:3],
    "Sweeping VEH": scores_sweep_V[:3],
    "Sweeping CNO": scores_sweep_C[:3],
}
for name, trajectory in scores.items():
    if not np.isfinite(trajectory).all():
        raise ValueError(f"Non-finite PCA scores: {name}")


### Plot PCA


In [ ]:
# ---------- Plot separate VEH vs CNO PCA figures ---------- #

savefig = True
show_ticks = False
show_legend = True

event_markers = {"stim_start": (STIM_START - PCA_START, "o")}

stimulus_trajectories = {
    "Looming": {
        "VEH": np.asarray(scores_loom_V, dtype=float)[:3],
        "CNO": np.asarray(scores_loom_C, dtype=float)[:3],
    },
    "Dimming": {
        "VEH": np.asarray(scores_dim_V, dtype=float)[:3],
        "CNO": np.asarray(scores_dim_C, dtype=float)[:3],
    },
    "Sweeping": {
        "VEH": np.asarray(scores_sweep_V, dtype=float)[:3],
        "CNO": np.asarray(scores_sweep_C, dtype=float)[:3],
    },
}

stimulus_colors = {
    "Looming": "indianred",
    "Dimming": "steelblue",
    "Sweeping": "mediumseagreen",
}


def plot_vehicle_cno_pca(stimulus_name, trajectories, color):
    for group, trajectory in trajectories.items():
        if trajectory.ndim != 2 or trajectory.shape[0] != 3:
            raise ValueError(
                f"{stimulus_name} {group} must have shape (3, time); "
                f"received {trajectory.shape}"
            )

    # Calculate common limits using both VEH and CNO.
    all_xyz = np.concatenate(list(trajectories.values()), axis=1)
    finite_samples = np.all(np.isfinite(all_xyz), axis=0)
    all_finite = all_xyz[:, finite_samples]

    if all_finite.shape[1] == 0:
        raise ValueError(f"No finite PCA samples found for {stimulus_name}.")

    data_min = all_finite.min(axis=1)
    data_max = all_finite.max(axis=1)
    data_mid = (data_min + data_max) / 2

    common_span = float(np.max(data_max - data_min))
    if common_span <= 0:
        common_span = 1.0

    half_span = 0.55 * common_span
    limits = np.column_stack((
        data_mid - half_span,
        data_mid + half_span,
    ))

    z_floor = limits[2, 0]
    y_wall = limits[1, 1]
    x_wall = limits[0, 0]

    fig = plt.figure(
        figsize=(55 / 25.4, 55 / 25.4),
        layout="constrained",
    )
    ax = fig.add_subplot(111, projection="3d")

    ax.set_xlim(*limits[0])
    ax.set_ylim(*limits[1])
    ax.set_zlim(*limits[2])
    ax.set_box_aspect((1, 1, 1))

    # Plot floor and wall projections.
    for group, trajectory in trajectories.items():
        x, y, z = trajectory
        linestyle = "-" if group == "VEH" else "--"

        ax.plot(
            x, y, np.full_like(z, z_floor),
            color=color,
            alpha=0.15,
            linewidth=0.6,
            linestyle=linestyle,
        )

        ax.plot(
            x, np.full_like(y, y_wall), z,
            color=color,
            alpha=0.15,
            linewidth=0.6,
            linestyle=linestyle,
        )

        ax.plot(
            np.full_like(x, x_wall), y, z,
            color=color,
            alpha=0.15,
            linewidth=0.6,
            linestyle=linestyle,
        )

    # Plot main trajectories and temporal markers.
    for group, trajectory in trajectories.items():
        x, y, z = trajectory
        linestyle = "-" if group == "VEH" else "--"

        ax.plot(
            x,
            y,
            z,
            color=color,
            linewidth=1.2,
            linestyle=linestyle,
            label=group,
        )

        for index, marker in [*event_markers.values(), (STIM_END[stimulus_name] - PCA_START, "^")]:
            if 0 <= index < trajectory.shape[1]:
                point = trajectory[:, index]

                if np.all(np.isfinite(point)):
                    ax.scatter(
                        point[0],
                        point[1],
                        point[2],
                        color=color,
                        marker=marker,
                        s=10,
                        depthshade=False,
                    )

    ax.set_xlabel(f"PC1 ({var_exp[0]:.1f}%)", fontsize=8, labelpad=-8)
    ax.set_ylabel(f"PC2 ({var_exp[1]:.1f}%)", fontsize=8, labelpad=-8)
    ax.set_zlabel(f"PC3 ({var_exp[2]:.1f}%)", fontsize=8, labelpad=-8)

    ax.set_title(stimulus_name, fontsize=9, fontweight="bold")
    ax.grid(False)

    if show_ticks:
        ax.tick_params(axis="x", labelsize=6, pad=-1)
        ax.tick_params(axis="y", labelsize=6, pad=-1)
        ax.tick_params(axis="z", labelsize=6, pad=-1)
    else:
        ax.set(xticks=[], yticks=[], zticks=[])

    if show_legend:
        ax.legend(
            loc="upper left",
            frameon=False,
            fontsize=7,
            title="Group",
            title_fontsize=7,
        )

    if savefig:
        filename = stimulus_name.lower().replace(" ", "_")
        save_figure(fig, f"PCA_{filename}_VEH_vs_CNO.png",
            dpi=300,
            bbox_inches="tight",
            facecolor="white",
        )
        save_figure(fig, f"PCA_{filename}_VEH_vs_CNO.svg",
            bbox_inches="tight",
            facecolor="white",
        )

    plt.show()
    plt.close(fig)


for stimulus_name, trajectories in stimulus_trajectories.items():
    plot_vehicle_cno_pca(
        stimulus_name,
        trajectories,
        stimulus_colors[stimulus_name],
    )


# Visual Brain AFFECTED Brain Regions


### Get Effect Sizes for Plotting


In [ ]:
response_values = {
    "Looming": res_CNO_LT['effect_size'],
    "Dimming": res_CNO_DT['effect_size'],
    "Sweeping": res_CNO_SB['effect_size'],    
}
region_values = {
    "Looming": res_CNO_LT['effect_size'],
    "Dimming": res_CNO_DT['effect_size'],
    "Sweeping": res_CNO_SB['effect_size'],
}

# p and q values
p_values = {
    "Looming": res_CNO_LT['p_raw'],
    "Dimming": res_CNO_DT['p_raw'],
    "Sweeping": res_CNO_SB['p_raw'],
}

q_values = {
    "Looming": res_CNO_LT['p_corrected'],
    "Dimming": res_CNO_DT['p_corrected'],
    "Sweeping": res_CNO_SB['p_corrected'],
}
for stimulus, values in region_values.items():
    print(stimulus, values.shape)


values_for_plot = {}
for stimulus, values in region_values.items():
    current = values.copy()
    if SIGNIFICANT_ONLY:
        current[~(q_values[stimulus] < FDR_ALPHA)] = np.nan
    values_for_plot[stimulus] = current

all_plot_values = np.concatenate(list(values_for_plot.values()))
finite_values = all_plot_values[np.isfinite(all_plot_values)]
color_limit = np.max(np.abs(finite_values)) if finite_values.size else 1.0
if color_limit == 0:
    color_limit = 1.0

cmap = sns.diverging_palette(220, 20, sep=30, as_cmap=True).with_extremes(
    bad=(1.0, 1.0, 1.0, 0.0)
)
selected_atlas_labels = np.unique(np.concatenate(selection.parts))
print(f"Symmetric color range: {-color_limit:.4g} to {color_limit:.4g}")


In [ ]:
ORIENTATION = "sagittal"

sagittal_fractions = [0.16, 0.29, 0.36, 0.4]

n_sagittal_planes = atlas_regions.shape[2]

plane_numbers = [
    int(round(1 + fraction * (n_sagittal_planes - 1)))
    for fraction in sagittal_fractions
]

slice_specs = [(f"Sagittal {f:.0%}", "fraction of atlas axis", f)
               for f in sagittal_fractions]

print("Sagittal plane numbers:", plane_numbers)


In [ ]:
# ---------- Separate compact 2×2 atlas figures ---------- #

stimuli = ["Looming", "Dimming", "Sweeping"]
width_mm = 85
height_mm = 78
savefig = True

# Exclude atlas root/background label when available.
root_labels = np.array([], dtype=int)

if "selection" in globals() and "root" in selection.acr:
    root_index = selection.acr.index("root")
    root_labels = np.asarray(selection.parts[root_index])

activity_cmap = cmap.with_extremes(
    bad=(1.0, 1.0, 1.0, 0.0)
)

silhouette_cmap = plt.colormaps["Greys"].with_extremes(
    bad=(1.0, 1.0, 1.0, 0.0)
)

orientation_name = ORIENTATION.lower()

for stimulus in stimuli:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(width_mm / 25.4, height_mm / 25.4),
        facecolor="white",
        constrained_layout=True,
    )

    axes = axes.ravel()
    fig.patch.set_facecolor("white")

    for ax, (slice_spec, plane_number) in zip(
        axes,
        zip(slice_specs, plane_numbers, strict=True),
        strict=True,
    ):
        title, target, _ = slice_spec

        label_slice = atlas_plane(
            atlas_regions,
            ORIENTATION,
            plane_number,
        )

        outside_mask = (
            (label_slice == 0)
            | np.isin(label_slice, root_labels)
        )

        tissue_mask = ~outside_mask
        selected_mask = (
            tissue_mask
            & np.isin(label_slice, selected_labels_without_root)
        )

        silhouette = np.ma.array(
            np.ones_like(label_slice, dtype=float),
            mask=~tissue_mask,
        )

        activity_slice = recolor_map_double(
            {"Regions": label_slice},
            selection,
            values_for_plot[stimulus],
        )

        activity_mask = (
            outside_mask
            | ~selected_mask
            | ~np.isfinite(activity_slice)
        )

        activity_masked = np.ma.array(
            activity_slice,
            mask=activity_mask,
        )

        ax.imshow(
            silhouette,
            cmap=silhouette_cmap,
            vmin=0,
            vmax=1.2,
            alpha=0.12,
            interpolation="nearest",
        )

        ax.imshow(
            activity_masked,
            cmap=activity_cmap,
            vmin=-color_limit,
            vmax=color_limit,
            interpolation="bilinear",
        )

        draw_borders(
            atlas,
            ORIENTATION,
            plane_number,
            ax=ax,
            color="white",
            linewidth=0,
        )

        ax.set_title(
            f"{title}\n{target}",
            fontsize=7,
            fontweight="bold",
            pad=2,
        )
        ax.set_axis_off()

    fig.suptitle(
        f"{stimulus} — {orientation_name.capitalize()}",
        fontsize=9,
        fontweight="bold",
    )

    mappable = ScalarMappable(
        norm=Normalize(-color_limit, color_limit),
        cmap=activity_cmap,
    )
    mappable.set_array([])

    cbar = fig.colorbar(
        mappable,
        ax=axes.tolist(),
        orientation="horizontal",
        fraction=0.035,
        pad=0.02,
        shrink=0.7,
    )

    cbar.set_ticks([-color_limit, 0, color_limit])
    cbar.ax.tick_params(labelsize=6)
    cbar.set_label("Cohen’s d", fontsize=7)
    cbar.outline.set_visible(False)

    if savefig:
        filename = stimulus.lower()

        save_figure(fig, f"CNO_effect_{orientation_name}_{filename}.png",
            dpi=300,
            bbox_inches="tight",
            facecolor="white",
        )
        save_figure(fig, f"CNO_effect_{orientation_name}_{filename}.svg",
            bbox_inches="tight",
            facecolor="white",
        )

    plt.show()
    plt.close("all")


# Plot Responses VEH vs CNO for each Stimulus


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ---------- Define the 4 Regions ---------- #
regions_to_plot = {
    "SCs": np.arange(242, 246),
    "SCi": np.arange(238, 243),
    "VISp(al)": np.array([37]),  # Original region index; verify against replacement labels.
    "LP": np.array([131])
}

# ---------- Setup Stimuli Data ---------- #
stimuli_configs = {
    "Loom": {
        "Vehicle": LoomV_smooth,
        "CNO": LoomC_smooth,
        "end_time": 41,
        "cno_color": "indianred" 
    },
    "Dimming": {
        "Vehicle": DimV_smooth,
        "CNO": DimC_smooth,
        "end_time": 41,
        "cno_color": "steelblue"
    },
    "Sweep": {
        "Vehicle": SweepV_smooth,
        "CNO": SweepC_smooth,
        "end_time": 46,
        "cno_color": "mediumseagreen"
    }
}

veh_color = "black"
baseline_start, baseline_end = BASELINE.start, BASELINE.stop
savefig = 1

# ---------- Setup Figure (Width 6, Height scalable) ---------- #
# 6 inches wide, 3 inches tall is a good aspect ratio for 4x3 sparklines
fig = plt.figure(figsize=(4.5, 2)) 
sns.set_theme(font=FONT_FAMILY, style="ticks")

# Create a 4 row x 3 column grid
gs = fig.add_gridspec(len(regions_to_plot), len(stimuli_configs), hspace=0.6, wspace=0.2)

# ---------- Plot the Grid ---------- #
for row_idx, (region_name, indices) in enumerate(regions_to_plot.items()):
    for col_idx, (stim_name, stim_info) in enumerate(stimuli_configs.items()):
        
        ax = fig.add_subplot(gs[row_idx, col_idx])
        valid_idx = np.setdiff1d(indices, NoplotIndexes)
        
        if len(valid_idx) > 0:
            local_min, local_max = float('inf'), float('-inf')
            
            # Plot Vehicle and CNO
            for condition in ["Vehicle", "CNO"]:
                data = stim_info[condition]
                raw_response = np.mean(data[valid_idx, :], axis=0)
                baseline = np.mean(raw_response[baseline_start:baseline_end])
                norm_response = raw_response - baseline
                
                color = veh_color if condition == "Vehicle" else stim_info["cno_color"]
                
                # Plot the trace
                ax.plot(norm_response, color=color, linewidth=1.2, label=condition)
                
                local_min = min(local_min, np.min(norm_response[20:70]))
                local_max = max(local_max, np.max(norm_response[20:70]))
                    
            local_range = local_max - local_min
            if local_range == 0: 
                local_range = 0.1
            ax.set_ylim(local_min - local_range * 0.1, local_max + local_range * 0.1)
                    
            # Stimulus timing lines
            ax.axvline(STIM_START, color=".4", linewidth=0.7)
            ax.axvline(stim_info["end_time"], color=stim_info["cno_color"], linewidth=0.7, linestyle="--", alpha=0.5)
            
            # Title formatting: 
            # Top row gets the stimulus name. Every plot gets the region name.
            if row_idx == 0:
                ax.set_title(f"{stim_name}\n{region_name}", fontweight="bold", fontsize=8, loc='left', pad=3)
            else:
                ax.set_title(region_name, fontweight="bold", fontsize=8, loc='left', pad=3)
            
        ax.set_xlim(20, 70)
        # Remove all axes, ticks, labels, and spines for sparkline aesthetic
        ax.axis('off')

fig.legend(handles=[plt.Line2D([0], [0], color="black", label="Vehicle"), plt.Line2D([0], [0], color="gray", label="CNO (stimulus color)")], loc="upper center", bbox_to_anchor=(0.5, 1.08), ncol=2, frameon=False, fontsize=6)

# Calibrated 2-second bar in sample coordinates on the final axes.
ax.set_xlim(20, 70)
ax.plot([60, 60 + 2 / SAMPLE_INTERVAL_S], [-0.12, -0.12],
        transform=ax.get_xaxis_transform(), color="black", clip_on=False)
ax.text(60 + 1 / SAMPLE_INTERVAL_S, -0.22, "2 s", ha="center",
        transform=ax.get_xaxis_transform(), fontsize=6)

if savefig == 1:
    save_figure(plt.gcf(), "Veh_vs_CNO_KeyRegions_Grid.png", dpi=300, bbox_inches="tight")
    save_figure(plt.gcf(), "Veh_vs_CNO_KeyRegions_Grid.svg", bbox_inches="tight")

plt.show()
plt.close("all")


In [ ]:
# ---------- Average VEH vs CNO responses for selected groups ---------- #
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Prefix matches include atlas subregions; PAGd/PAGv combine verified dorsal/ventral labels.
groups_to_plot = ["SCs", "SCi", "SCd", "PAGd", "PAGv", "LGv", "LGd", "LP"]
group_patterns = {
    "SCs": ("SCs",),
    "SCi": ("SCi",),
    "SCd": ("SCd",),
    "PAGd": ("PAGmd", "PAGpd"),
    "PAGv": ("PAGmv", "PAGpv"),
    "LGv": ("LGv",),
    "LGd": ("LGd",),
    "LP": ("LP",),
}

region_labels_array = np.asarray(region_labels, dtype=object)
region_labels_lower = np.char.lower(region_labels_array.astype(str))
group_indices = {}
missing_groups = []

for group_name in groups_to_plot:
    patterns = group_patterns[group_name]
    if group_name == "LP":
        matches = np.flatnonzero(region_labels_lower == "lp")
    else:
        matches = np.flatnonzero(
            np.logical_or.reduce(
                [np.char.startswith(region_labels_lower, pattern.lower()) for pattern in patterns]
            )
        )
    if matches.size == 0:
        missing_groups.append(group_name)
    else:
        group_indices[group_name] = matches

if missing_groups:
    raise ValueError(f"Requested groups were not found: {missing_groups}")

print("Groups and region indices:")
for group_name, indices in group_indices.items():
    print(f"{group_name}: {region_labels_array[indices].tolist()} -> {indices.tolist()}")

stimuli_configs = {
    "Looming": (LoomV_smooth, LoomC_smooth, 41, "indianred"),
    "Dimming": (DimV_smooth, DimC_smooth, 41, "steelblue"),
    "Sweeping": (SweepV_smooth, SweepC_smooth, 46, "mediumseagreen"),
}

baseline_start, baseline_end = BASELINE.start, BASELINE.stop
plot_start, plot_end = 20, 70
sns.set_theme(font=FONT_FAMILY, style="ticks")
fig, axes = plt.subplots(
    len(groups_to_plot),
    len(stimuli_configs),
    figsize=(6.5, 10),
    sharex=True,
    squeeze=False,
)

for row_idx, group_name in enumerate(groups_to_plot):
    indices = group_indices[group_name]
    for col_idx, (stimulus_name, (vehicle_data, cno_data, end_time, cno_color)) in enumerate(stimuli_configs.items()):
        ax = axes[row_idx, col_idx]
        traces = {}
        for condition, data in (("Vehicle", vehicle_data), ("CNO", cno_data)):
            response = np.mean(data[indices, :], axis=0)
            baseline = np.mean(response[baseline_start:baseline_end])
            traces[condition] = response - baseline

        local_values = np.concatenate([trace[plot_start:plot_end] for trace in traces.values()])
        local_min, local_max = float(np.min(local_values)), float(np.max(local_values))
        local_range = max(local_max - local_min, 0.1)
        ax.set_ylim(local_min - 0.1 * local_range, local_max + 0.1 * local_range)
        ax.plot(traces["Vehicle"], color="black", linewidth=1.0, label="Vehicle")
        ax.plot(traces["CNO"], color=cno_color, linewidth=1.0, linestyle="--", label="CNO")
        ax.axvline(STIM_START, color=".4", linewidth=0.7)
        ax.axvline(end_time, color=cno_color, linewidth=0.7, linestyle=":", alpha=0.7)
        ax.set_xlim(plot_start, plot_end)
        ax.set_title(f"{stimulus_name}\n{group_name}", fontsize=8, fontweight="bold", loc="left", pad=3)
        ax.set_xticks([20, 30, 60])
        ax.set_xticklabels([f"{(frame - STIM_START) * SAMPLE_INTERVAL_S:g}" for frame in [20, 30, 60]], fontsize=6)
        ax.tick_params(axis="y", labelsize=6)
        ax.set_xlabel("Time (s)", fontsize=7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

axes[0, 0].legend(frameon=False, fontsize=7, loc="upper left", handlelength=1.8)
fig.supylabel("Baseline-subtracted response", fontsize=8)
fig.tight_layout()

if globals().get("savefig", True):
    save_figure(fig, "Average_Group_Responses_VEH_vs_CNO.png", dpi=300, bbox_inches="tight")
    save_figure(fig, "Average_Group_Responses_VEH_vs_CNO.svg", bbox_inches="tight")

plt.show()
plt.close("all")


## Export statistics and record provenance

Tables contain all 319 regions, with missing adjusted p-values for regions excluded from FDR. The manifest records input checksums, analysis settings, and the software versions used.


In [ ]:
def export_statistics(results, prefix):
    for stimulus, result in results.items():
        table = pd.DataFrame({"region_index": np.arange(319), "region": region_labels,
                              **{key: value for key, value in result.items() if key != "sig_rows"}})
        table.to_csv(OUTPUT_DIR / f"{prefix}_{stimulus}.csv", index=False)

export_statistics({"LT": res_LT, "DT": res_DT, "SB": res_SB}, "activity")
for tail, results in cno_by_tail.items():
    export_statistics(results, f"cno_{tail}")
shared_response.to_csv(OUTPUT_DIR / "shared_activity_summary.csv", index=False)
stats_df.to_csv(OUTPUT_DIR / "panel_C_dabest.csv", index=False)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "packages": {p: importlib.metadata.version(p) for p in
                 ["numpy", "pandas", "scipy", "matplotlib", "seaborn", "scikit-learn", "statsmodels", "dabest", "h5py"]},
    "inputs": [{"file": p.name, "sha256": sha256(p)} for p in input_files],
    "input_blocks": input_manifest,
    "source_files": {name: sha256(PROJECT_DIR / name) for name in
                     ["fUSI_FIGURE_03_reproducible.ipynb", "atlas_tools.py", "analysis_helpers.py", "requirements.txt"]},
    "settings": {"seed": RANDOM_SEED, "permutations": N_PERMUTATIONS,
                 "activity_bootstraps": N_BOOTSTRAPS_ACTIVITY, "cno_bootstraps": N_BOOTSTRAPS_CNO,
                 "baseline": [BASELINE.start, BASELINE.stop],
                 "activity_window": [ACTIVITY_WINDOW.start, ACTIVITY_WINDOW.stop],
                 "cno_window": [CNO_WINDOW.start, CNO_WINDOW.stop],
                 "fdr_alpha": FDR_ALPHA, "sample_interval_s": SAMPLE_INTERVAL_S,
                 "paired_animals_confirmed": PAIRED_ANIMALS_CONFIRMED,
                 "cno_plot_tail": "less", "pca_solver": "full"},
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"Completed: figures, tables, and run_manifest.json saved to {OUTPUT_DIR}")
